<a href="https://colab.research.google.com/github/anushkag0211/WhatInTheRepo/blob/main/WhatInRepo1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
!pip install openai faiss-cpu sentence-transformers gitpython transformers accelerate

In [17]:
from git import Repo

repo_url = "https://github.com/psf/requests"
repo_path = "/content/repo"

Repo.clone_from(repo_url, repo_path)
print("Repo downloaded!")

Repo downloaded!


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [19]:
import os

def load_files(path):
    files_data = []

    for root, _, files in os.walk(path):
        for file in files:
            if file.endswith(".py"):
                file_path = os.path.join(root, file)

                try:
                    with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
                        files_data.append({
                            "file": file_path.replace(path, ""),
                            "text": f.read()
                        })
                except:
                    pass

    return files_data

docs = load_files(repo_path)
print("Files loaded:", len(docs))

Files loaded: 36


In [20]:
def chunk_text(text, chunk_size=300):
    return [text[i:i+chunk_size] for i in range(0, len(text), chunk_size)]

chunks = []

for doc in docs:
    for chunk in chunk_text(doc["text"]):
        chunks.append({
            "file": doc["file"],
            "text": chunk
        })

print("Total chunks:", len(chunks))

Total chunks: 641


In [21]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [22]:
texts = [c["text"] for c in chunks]
embeddings = embedding_model.encode(texts, show_progress_bar=True)

Batches:   0%|          | 0/21 [00:00<?, ?it/s]

In [25]:
import faiss
import numpy as np

dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)

index.add(np.array(embeddings))

In [26]:
from transformers import pipeline

llm = pipeline(
    "text-generation",
    model="microsoft/Phi-3-mini-4k-instruct",
    device=0  # GPU
)

config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/599 [00:00<?, ?B/s]

In [37]:
def retrieve(query, k=2):
    query_emb = embedding_model.encode([query])
    distances, indices = index.search(np.array(query_emb), k)

    return [chunks[i] for i in indices[0]]

In [38]:
def ask_llm(prompt):
    out = llm(
        prompt,
        max_new_tokens=300,
        temperature=0.2,
        do_sample=True,
        return_full_text=False   # 🔥 IMPORTANT FIX
    )

    return out[0]["generated_text"].strip()

In [41]:
def ask_llm_rag(query, retrieved_chunks):

    context = ""

    for c in retrieved_chunks:
        context += f"\nFile: {c['file']}\nCode:\n{c['text']}\n---\n"

    prompt = f"""
You are a senior software engineer.

Answer clearly in 5-7 lines maximum.

Do NOT repeat code.

Do NOT stop mid-sentence.

If unsure, say "Not enough information."

QUESTION:
{query}

CODE:
{context}

ANSWER:
"""

    return ask_llm(prompt)

In [42]:
def ask_codebase(query):
    retrieved = retrieve(query)
    answer = ask_llm_rag(query, retrieved)

    return {
        "question": query,
        "answer": answer,
        "sources": list(set([r["file"] for r in retrieved]))
    }

In [43]:
result = ask_codebase("How does error handling work in this repo?")

print("ANSWER:\n", result["answer"])
print("\nSOURCES:\n", result["sources"])

Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


ANSWER:
 In the provided code snippet from the `models.py` file within the `src/requests` directory, the `ok` property is defined to determine if the HTTP response status code indicates a successful request. The `ok` property returns `True` if the status code is less than 400, which includes the range of successful HTTP status codes (200-299). If the status code is 400 or above, it indicates a client or server error, and the `ok` property will return `False`. The `__nonzero__` method is a Python 2.x compatibility method that allows the object to be used in a boolean context, returning `True` if `ok` is `True` and `False` otherwise. The `__iter__` method enables the response object to be iterable, allowing the caller to iterate over the content of the response in chunks of 128 bytes.

SOURCES:
 ['/src/requests/models.py']
